# DeepLabV3+ Training & Inference Notebook (Binary Leaf Segmentation)

Notebook ini untuk melatih model DeepLabV3+ untuk **segmentasi biner daun**
(0 = background, 1 = leaf) pada dataset Plant Phenotyping, mengikuti rancangan
`docs/ensembel.md` agar hasilnya dapat di-ensembel dengan model U-Net yang sudah
dilatih.

**Environment:** VS Code + Colab Kernel (GPU Colab)
**Dataset:** Downloaded via `data/download_dataset.py` (kagglehub)
**Model:** DeepLabV3+ dari `models/deeplab.py` dengan backbone ResNet/Xception/DRN/MobileNet
**Output:** Checkpoint `.pth.tar` di folder `experiments/`
**Ensemble:** `ensemble/fuse_predictions.py` menggabungkan probabilitas daun U-Net + DeepLabV3

## 0. Setup Environment (Colab-specific)

In [1]:
import os, shutil, sys
from pathlib import Path

REPO_URL = 'https://github.com/adinmusababa/segmentasi.git'
REPO_DIR = Path('/content/segmentasi')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
!git clone -b setup {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
print(f'CWD: {os.getcwd()}')

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

REPO_PATH = REPO_DIR  # alias global untuk sel-sel berikutnya
IN_COLAB = True

Cloning into '/content/segmentasi'...
remote: Enumerating objects: 162, done.
remote: Counting objects: 100% (162/162), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 162 (delta 64), reused 149 (delta 51), pack-reused 0 (from 0)
Receiving objects: 100% (162/162), 1.58 MiB | 723.00 KiB/s, done.
Resolving deltas: 100% (64/64), done.
/content/segmentasi
CWD: /content/segmentasi


In [5]:
# Download dataset
!python data/download_dataset.py

Plant Phenotyping Dataset Downloader

Downloading... (this may take a while for large datasets)
Using Colab cache for faster access to the 'plant-phenotyping-dataset' dataset.
Dataset downloaded to: /kaggle/input/plant-phenotyping-dataset
Dataset root: /kaggle/input/plant-phenotyping-dataset/Plant_Phenotyping_Datasets

Organizing dataset into data/imgs/ and data/masks/

  Processing: Plant/Ara2012
    Found 120 RGB images
    Copied: 120 image/mask pairs, Skipped: 0 (no label)

  Processing: Plant/Ara2013-Canon
    Found 165 RGB images
    Copied: 165 image/mask pairs, Skipped: 0 (no label)

  Processing: Plant/Tobacco
    Found 62 RGB images
    Copied: 62 image/mask pairs, Skipped: 0 (no label)

  Total: 347 image/mask pairs copied

Verifying dataset
  Images: 347
  Masks:  347
  Matched pairs: 347

Done! Dataset is ready for training.
  Images: /content/segmentasi/data/imgs  (347 files)
  Masks:  /content/segmentasi/data/masks  (347 files)
  Matched pairs: 347

To train the model, r

In [ ]:
# # Colab environment setup
# import sys
# import os
# from pathlib import Path

# # Detect if running in Colab
# IN_COLAB = 'google.colab' in sys.modules
# print(f"IN_COLAB: {IN_COLAB}")

# # NOTE: Semua data (repo + dataset) di-simpan di session runtime Colab (/content),
# # yang bersifat sementara dan hilang saat runtime di-reset/disconnect.
# # Ini sesuai preferensi: TIDAK menyimpan ke Google Drive.
# # Kalau mau file hasil (dataset, checkpoint) tahan lama, simpan manual ke Drive.

# if IN_COLAB:
#     # Clone repo ke session runtime (bukan Drive)
#     REPO_PATH = Path('/content/deeplabV3-PyTorch')
#     if not REPO_PATH.exists():
#         print("Cloning repository...")
#         !git clone https://github.com/adinmusababa/deeplabV3-PyTorch.git /content/deeplabV3-PyTorch
#     os.chdir(REPO_PATH)
#     print(f"Working dir: {os.getcwd()}")
# else:
#     # Local/VS Code: assume already in repo root
#     REPO_PATH = Path.cwd()
#     while not (REPO_PATH / 'models').exists() and REPO_PATH != REPO_PATH.parent:
#         REPO_PATH = REPO_PATH.parent
#     os.chdir(REPO_PATH)
#     print(f"Working dir: {os.getcwd()}")

# # Add project root to sys.path
# if str(REPO_PATH) not in sys.path:
#     sys.path.insert(0, str(REPO_PATH))

# # Verify structure
# print("models/ exists:", (REPO_PATH / "models").exists())
# print("data/ exists:", (REPO_PATH / "data").exists())
# print("configs/ exists:", (REPO_PATH / "configs").exists())
# print("kagglehub cache di: /root/.cache/kagglehub (session temp, bukan Drive)")

## 1. Install Dependencies

In [2]:
# Install requirements
!pip install -q kagglehub pyyaml tensorboardX tqdm scikit-learn matplotlib pillow numpy torch torchvision

# Verify torch CUDA
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


## 2. Download & Organize Dataset

In [ ]:
# # Run download script
# import subprocess
# result = subprocess.run([sys.executable, "data/download_dataset.py"], capture_output=True, text=True)
# print(result.stdout)
# if result.stderr:
#     print("STDERR:", result.stderr)

# # Verify
# from pathlib import Path
# imgs = list((REPO_PATH / "data" / "imgs").glob("*.png"))
# masks = list((REPO_PATH / "data" / "masks").glob("*.png"))
# print(f"Images: {len(imgs)}")
# print(f"Masks: {len(masks)}")
# if imgs:
#     print(f"First few: {[f.name for f in imgs[:5]]}")

## 3. Configure Training (EDIT HERE)

In [ ]:
import torch
import yaml
from pathlib import Path

# Load base config
with open("configs/config.yml") as f:
    config = yaml.safe_load(f)

# ========== OVERRIDE FOR BINARY LEAF SEGMENTATION (ENSEMBLE WITH U-NET) ==========
config["dataset"]["base_path"] = str(REPO_PATH)  # root project
config["dataset"]["dataset_name"] = "plant_phenotyping"
# Binary leaf segmentation: background(0) + leaf(1) — must match U-Net
config["network"]["num_classes"] = 2
config["network"]["backbone"] = "resnet"  # pilihan: resnet, xception, drn, mobilenet
config["network"]["sync_bn"] = False  # True hanya kalau multi-GPU
config["network"]["freeze_bn"] = True  # batch kecil -> BatchNorm tidak stabil
config["network"]["use_cuda"] = torch.cuda.is_available()

config["image"]["out_stride"] = 16
config["image"]["base_size"] = 256  # harus sama dengan U-Net
config["image"]["crop_size"] = 256  # harus sama dengan U-Net

config["training"]["workers"] = 4 if torch.cuda.is_available() else 0
config["training"]["batch_size"] = 4 if torch.cuda.is_available() else 2
config["training"]["epochs"] = 50  # ubah sesuai kebutuhan
config["training"]["start_epoch"] = 0
config["training"]["lr"] = 0.0005
config["training"]["lr_scheduler"] = "poly"  # poly, step, cos
config["training"]["momentum"] = 0.9
config["training"]["weight_decay"] = 0.0005
config["training"]["nesterov"] = False
config["training"]["loss_type"] = "ce_dice"  # ce, focal, ce_dice (rekomendasi)
config["training"]["use_balanced_weights"] = True  # background >> leaf
config["training"]["no_val"] = False
config["training"]["val_interval"] = 1
config["training"]["train_on_subset"]["enabled"] = False  # training final pakai seluruh train set
config["training"]["train_on_subset"]["dataset_fraction"] = 1.0

# Resume training (optional)
config["training"]["weights_initialization"]["use_pretrained_weights"] = False  # True kalau mau resume
config["training"]["weights_initialization"]["restore_from"] = "./experiments/checkpoint_last.pth.tar"

config["training"]["tensorboard"]["enabled"] = True
config["training"]["tensorboard"]["log_dir"] = "./tensorboard/"

# Seed & split identik dengan U-Net agar ensemble fair
config["seed"] = 42

# Save modified config
config_path = REPO_PATH / "configs" / "config_plant.yml"
with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Config saved to: {config_path}")
print("Key settings:")
print(f"  num_classes: {config["network"]["num_classes"]}")
print(f"  backbone: {config["network"]["backbone"]}")
print(f"  batch_size: {config["training"]["batch_size"]}")
print(f"  epochs: {config["training"]["epochs"]}")
print(f"  crop_size: {config["image"]["crop_size"]}")
print(f"  loss_type: {config["training"]["loss_type"]}")
print(f"  use_balanced_weights: {config["training"]["use_balanced_weights"]}")
print(f"  use_cuda: {config["network"]["use_cuda"]}")

## 4. Training

In [4]:
# Import Trainer
from trainers.trainer import Trainer

# Buat direktori experiments jika belum ada (untuk checkpoint)
(REPO_PATH / "experiments").mkdir(parents=True, exist_ok=True)

# checkname diperlukan oleh Trainer/Saver
config["checkname"] = "deeplab-" + str(config["network"]["backbone"])

# Initialize trainer
trainer = Trainer(config)

print(f"Starting Epoch: {trainer.config["training"]["start_epoch"]}")
print(f"Total Epochs: {trainer.config["training"]["epochs"]}")
print(f"Train loader: {len(trainer.train_loader)} batches")
print(f"Val loader: {len(trainer.val_loader)} batches")
print(f"Test loader: {len(trainer.test_loader)} batches")
print(f"Classes: {trainer.nclass}")

RuntimeError: Could not find dataset dirs. Put images in data/imgs/ and color masks in data/masks/ or update config.dataset.base_path

In [ ]:
# Run training loop
for epoch in range(trainer.config['training']['start_epoch'], trainer.config['training']['epochs']):
    trainer.training(epoch)
    if not trainer.config['training']['no_val'] and epoch % config['training']['val_interval'] == (config['training']['val_interval'] - 1):
        trainer.validation(epoch)

trainer.writer.close()
print("Training completed!")

## 5. Load Best Model for Inference

In [ ]:
# Load predictor with best checkpoint
from predictors.predictor import Predictor

checkpoint_path = './experiments/checkpoint_best.pth.tar'
if not Path(checkpoint_path).exists():
    checkpoint_path = './experiments/checkpoint_last.pth.tar'
    print(f"Best not found, using last: {checkpoint_path}")
else:
    print(f"Using best checkpoint: {checkpoint_path}")

predictor = Predictor(config, checkpoint_path=checkpoint_path)
print(f"Model loaded. Classes: {predictor.num_classes}")

## 6. Inference on Single Image

In [ ]:
# Test on a sample image from dataset
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Check prerequisites
if "REPO_PATH" not in globals():
    raise RuntimeError("REPO_PATH not defined. Run Cell 2 (clone repo) first.")
if "predictor" not in globals():
    raise RuntimeError("predictor not defined. Run Cell 15 (load predictor) first.")

# Pick first image from data/imgs
imgs_dir = REPO_PATH / "data" / "imgs"
if not imgs_dir.exists():
    raise RuntimeError(f"data/imgs not found at {imgs_dir}. Run Cell 3 (download dataset) first.")

test_images = list(imgs_dir.glob("*.png"))
if not test_images:
    raise RuntimeError(f"No PNG images found in {imgs_dir}. Run Cell 3 (download dataset) first.")

test_img = str(test_images[0])
print(f"Testing on: {test_img}")
print(f"Total images available: {len(test_images)}")

# predict_probability() -> foreground (leaf) probability, untuk ensemble
prob = predictor.predict_probability(test_img)
image, pred_mask = predictor.segment_image(test_img)

# Visualize (binary: 0=background, 1=leaf)
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(image.astype(np.uint8))
axes[0].set_title("Original Image")
axes[0].axis('off')

axes[1].imshow(prob, cmap="hot", vmin=0, vmax=1)
axes[1].set_title("Leaf Probability (softmax[:,1])")
axes[1].axis('off')

axes[2].imshow(pred_mask, cmap="gray", vmin=0, vmax=1)
axes[2].set_title("Binary Mask")
axes[2].axis('off')

# Overlay
overlay = image.copy()
green = np.zeros_like(image)
green[:, :, 1] = 200
mask_bool = pred_mask.astype(bool)
overlay[mask_bool] = (0.5 * overlay[mask_bool] + 0.5 * green[mask_bool]).astype(np.uint8)
axes[3].imshow(overlay)
axes[3].set_title("Overlay (leaf=green)")
axes[3].axis('off')

plt.tight_layout()
plt.show()

print(f"Prediction shape: {pred_mask.shape}")
print(f"Unique classes predicted: {np.unique(pred_mask)}")
print(f"Leaf fraction: {pred_mask.mean():.4f}")

In [ ]:
# Plot training history dari TensorBoard logs
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

TENSORBOARD_DIR = REPO_PATH / "tensorboard"

# Collect all scalars from all event files
ea = EventAccumulator(str(TENSORBOARD_DIR), size_warning=False)
ea.Reload()

tags = ea.Tags()["scalars"]
print(f"Found {len(tags)} scalar tags: {tags}")

# Build a dict of tag -> (steps, values)
data = {}
for tag in tags:
    events = ea.Scalars(tag)
    steps = [e.step for e in events]
    vals = [e.value for e in events]
    data[tag] = (steps, vals)

# Plot combined training history
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("DeepLabV3+ Training History (Binary Leaf)", fontsize=16, fontweight="bold")

# 1. Training & Validation Loss
ax = axes[0, 0]
if "train/total_loss_epoch" in data:
    steps, vals = data["train/total_loss_epoch"]
    ax.plot(steps, vals, "b-", label="Train Loss", linewidth=2)
if "val/total_loss_epoch" in data:
    steps, vals = data["val/total_loss_epoch"]
    ax.plot(steps, vals, "r-", label="Val Loss", linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training & Validation Loss")
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Foreground Dice & IoU (leaf) — metrik utama untuk ensemble
ax = axes[0, 1]
if "val/Dice_leaf" in data:
    steps, vals = data["val/Dice_leaf"]
    ax.plot(steps, vals, "g-", label="Val Dice (leaf)", linewidth=2)
if "val/IoU_leaf" in data:
    steps, vals = data["val/IoU_leaf"]
    ax.plot(steps, vals, "y-", label="Val IoU (leaf)", linewidth=2)
if "val/mIoU" in data:
    steps, vals = data["val/mIoU"]
    ax.plot(steps, vals, "m--", label="mIoU", linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Metric")
ax.set_title("Foreground Dice / IoU (leaf)")
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Accuracy
ax = axes[1, 0]
if "val/Acc" in data:
    steps, vals = data["val/Acc"]
    ax.plot(steps, vals, "c-", label="Overall Acc", linewidth=2)
if "val/Acc_class" in data:
    steps, vals = data["val/Acc_class"]
    ax.plot(steps, vals, "m-", label="Mean Acc", linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy")
ax.legend()
ax.grid(True, alpha=0.3)

# 4. FWIoU
ax = axes[1, 1]
if "val/fwIoU" in data:
    steps, vals = data["val/fwIoU"]
    ax.plot(steps, vals, "y-", label="FWIoU", linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Frequency Weighted IoU")
ax.set_title("Frequency Weighted IoU")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Training history plotted successfully!")

## 7. Batch Inference on Test Set (Evaluation)

In [ ]:
# Run evaluation on test set
predictor.inference_on_test_set()

## 8. Batch Inference on Folder (Save Predictions)

In [ ]:
# Save predictions for all images in a folder
from pathlib import Path
from tqdm import tqdm

INPUT_DIR = REPO_PATH / "data" / "imgs"
OUTPUT_DIR = REPO_PATH / "inference_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

img_files = sorted(list(INPUT_DIR.glob("*.png")))
print(f"Processing {len(img_files)} images...")

for img_path in tqdm(img_files):
    try:
        # binary mask 0/1
        _, prediction = predictor.segment_image(str(img_path))
        out_path = OUTPUT_DIR / f"{img_path.stem}_pred.png"
        Image.fromarray(prediction.astype(np.uint8)).save(out_path)

        # foreground probability (untuk ensemble dengan U-Net)
        prob = predictor.predict_probability(str(img_path))
        prob_out = OUTPUT_DIR / f"{img_path.stem}_prob.npy"
        np.save(prob_out, prob)
    except Exception as e:
        print(f"Error on {img_path.name}: {e}")

print(f"Done! Masks in: {OUTPUT_DIR} (*_pred.png) and probs (*_prob.npy)")

## 9. TensorBoard (Optional)

In [ ]:
# Launch TensorBoard in Colab
if IN_COLAB:
    %load_ext tensorboard
    %tensorboard --logdir ./tensorboard --port 6006
else:
    print("Run locally: tensorboard --logdir ./tensorboard")

## 10. Tips & Next Steps

- **Ensemble dengan U-Net:** gunakan `ensemble/fuse_predictions.py` (load U-Net + DeepLabV3, weighted average probabilitas daun)
- **Cepatkan eksperimen:** turunkan `epochs` ke 5-10, atau aktifkan `train_on_subset`
- **Ganti backbone:** `mobilenet` atau `xception` lebih cepat dari `resnet`
- **Resume training:** set `weights_initialization.use_pretrained_weights: true` dan `start_epoch`
- **Class weights:** `use_balanced_weights: true` (background >> leaf) sudah aktif di config
- **freeze_bn:** `true` untuk batch kecil; test `false` hanya jika BatchNorm stabil
- **Multi-GPU:** set `sync_bn: true` dan `use_cuda: true`
- **Checkpoint:** `best_pred` dipilih berdasarkan Dice foreground (leaf) di validation set, bukan test set
- **Split:** identik dengan U-Net via `splits/{train,val,test}.txt`